# GMaster nside ladder on a Kaggle GPU

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/licongxu/GMaster/blob/cursor/kaggle-demo-1a91/examples/gmaster_kaggle_nside_sweep.ipynb)

Same estimator as the 1-1 demo (`NmtField(n_iter=3)`, coupling matrix, coupled cell, decouple;
spin 0; galactic cut + C1; `lmax = 3 nside - 1`; `NmtBin.from_lmax_linear(lmax, 50)`).

Times **GMaster on one T4** against **NaMaster on the session CPUs** at
`nside = 64, 128, 256, 512, 1024, 2048, 4096`. Kaggle: **Accelerator = GPU T4 x2**, **Internet on**,
then Run All. The install pins `CUDA_VISIBLE_DEVICES=0`.


In [ ]:
#@title 1. Install (Colab / Kaggle; ~5 min, `pymaster` and `fitsio` build from source) { display-mode: "form" }
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_KAGGLE = os.path.isdir("/kaggle/working") or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
IN_HOSTED = IN_COLAB or IN_KAGGLE

# One GPU for the 1-1 timing. Kaggle "GPU T4 x2" otherwise exposes two devices
# and the default jax calculator shards scalar SHTs at L>=2048.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# This branch: act-dr6-demo does not contain this notebook or the s2fft pin.
GMASTER_REF = "cursor/kaggle-demo-1a91"  #@param {type:"string"}
# PyPI s2fft 1.4.0 does `from s2fft_lib import _s2fft` at import and crashes on
# Colab Python 3.13 (the extension is a namespace without `_s2fft`). This SHA
# has the optional import (#362) and `_ftm_flm_primitive` (#380).
S2FFT_GIT = "git+https://github.com/astro-informatics/s2fft.git@86a4dff303539202ebbfa28661d85331e0e2bff4"


def sh(cmd, extra_env=None):
    print("$", cmd, flush=True)
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, env=env)
    if r.returncode:
        print(r.stdout[-4000:], r.stderr[-4000:])
        raise RuntimeError(f"failed: {cmd}")


if IN_HOSTED:
    sh("apt-get update -qq > /dev/null && "
       "apt-get install -y -qq autoconf automake libtool libgsl-dev libfftw3-dev libcfitsio-dev > /dev/null")
    print("pymaster builds from source here (~3 min) ...", flush=True)
    sh(f"{sys.executable} -m pip install -q camb psutil healpy pymaster")
    subprocess.run(f"{sys.executable} -m pip uninstall -y s2fft s2fft-lib s2fft_lib",
                   shell=True, capture_output=True)
    print("installing s2fft from git, CPU extension only (host nvcc would try extra sm_XX) ...", flush=True)
    # Hide nvcc so CMake takes the NO_CUDA_COMPILER branch and still produces s2fft_lib._s2fft.
    sh(f"{sys.executable} -m pip install -q --upgrade --no-cache-dir {S2FFT_GIT}",
       extra_env={"CMAKE_CUDA_COMPILER": "/does/not/exist"})
    sh(f"{sys.executable} -m pip install -q --upgrade --no-cache-dir git+https://github.com/licongxu/GMaster.git@{GMASTER_REF}")
    import importlib
    from importlib.metadata import version
    for k in list(sys.modules):
        if k == "s2fft" or k.startswith("s2fft.") or k.startswith("s2fft_lib"):
            del sys.modules[k]
    for mod in ("s2fft", "s2fft.utils.healpix_ffts", "s2fft.transforms._ftm_flm_primitive", "gmaster"):
        importlib.import_module(mod)
        print("imported", mod)
    print({p: version(p) for p in ("jax", "pymaster", "gmaster", "s2fft", "healpy", "camb")})
    print("host:", "Kaggle" if IN_KAGGLE else "Colab")
    print("If a previous cell already imported s2fft 1.4.0: restart the session, then run from cell 2.")
else:
    print("Not in Colab/Kaggle: assuming gmaster, pymaster, camb, healpy, psutil are already installed.")


In [ ]:
#@title nside ladder: GMaster (one T4) vs NaMaster (CPU)
import gc
import os
import shutil
import subprocess
import time
import traceback

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import healpy as hp
import numpy as np
import psutil
import pymaster as nmt
import gmaster as gm
from gmaster import _march_v2
from gmaster._cuda_gpu import is_cuda_device_kind, on_cuda_gpu
from gmaster._spin_march_pallas import fold_requested
from gmaster.utils import _use_pallas_sht
from gmaster.workspaces import _TT_QUADRATURE_LMAX

NSIDES = [64, 128, 256, 512, 1024, 2048, 4096]
N_ITER = 3
NLB = 50
GAL_CUT_DEG = 20.0
APO_DEG = 2.0
SEED = 1234
SPIN = 0

dev = jax.devices()[0]
PLATFORM = dev.platform
DEVICE_KIND = getattr(dev, "device_kind", PLATFORM)
NCORES = len(os.sched_getaffinity(0))
HOST_GB = psutil.virtual_memory().total / 1e9
nvcc = os.environ.get("GMASTER_NVCC") or shutil.which("nvcc") or (
    "/usr/local/cuda/bin/nvcc" if os.path.exists("/usr/local/cuda/bin/nvcc") else None)
print(f"JAX {jax.__version__}  backend={PLATFORM}  device={DEVICE_KIND}")
print(f"CPU cores={NCORES}  host RAM={HOST_GB:.1f} GB  nvcc={nvcc}  cuda_gpu={on_cuda_gpu()}  "
      f"kind_ok={is_cuda_device_kind(DEVICE_KIND)}", flush=True)
if PLATFORM != "gpu":
    raise RuntimeError(f"Need a CUDA GPU for this ladder, got {PLATFORM}/{DEVICE_KIND}")
print("building v2 march (first use only) ...", flush=True)
print("v2 CUDA march library:", "enabled" if _march_v2.enabled(192) else f"UNAVAILABLE: {_march_v2.unavailable_reason()}")

rng = np.random.default_rng(SEED)


def gb(x):
    return f"{x / 1e9:.2f} GB"


def smi_used():
    q = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"],
        capture_output=True, text=True)
    if q.returncode != 0 or not q.stdout.strip():
        return None
    return float(q.stdout.split()[0]) * 1024 ** 2


def make_maps(nside):
    lmax = 3 * nside - 1
    npix = 12 * nside * nside
    ell = np.arange(lmax + 1)
    cl = np.zeros(lmax + 1)
    cl[2:] = 1.0 / (ell[2:] * (ell[2:] + 1.0))
    t = hp.synfast(cl, nside, lmax=lmax, new=True)
    pix_arcmin2 = hp.nside2pixarea(nside, degrees=True) * 3600.0
    sigma = 10.0 / np.sqrt(pix_arcmin2)
    maps = [t + sigma * rng.standard_normal(npix)]
    theta, _ = hp.pix2ang(nside, np.arange(npix))
    b_deg = 90.0 - np.degrees(theta)
    mask = (np.abs(b_deg) > GAL_CUT_DEG).astype(np.float64)
    mask = nmt.mask_apodization(mask, APO_DEG, apotype="C1")
    return lmax, maps, np.asarray(mask, dtype=np.float64)


def gmaster_once(mask, maps, bins):
    t0 = time.perf_counter()
    f = gm.NmtField(mask, maps, n_iter=N_ITER)
    jax.block_until_ready(f.get_alms())
    t_field = time.perf_counter() - t0
    t1 = time.perf_counter()
    ws = gm.NmtWorkspace()
    ws.compute_coupling_matrix(f, f, bins)
    jax.block_until_ready(ws.mcm_binned)
    t_coup = time.perf_counter() - t1
    t2 = time.perf_counter()
    pcl = gm.compute_coupled_cell(f, f)
    jax.block_until_ready(pcl)
    t_pcl = time.perf_counter() - t2
    t3 = time.perf_counter()
    cl = np.asarray(jax.block_until_ready(ws.decouple_cell(pcl)))
    t_dec = time.perf_counter() - t3
    return cl, {"field": t_field, "coupling": t_coup, "coupled_cell": t_pcl, "decouple": t_dec}


def namaster_once(mask, maps, bins):
    f = nmt.NmtField(mask, maps, n_iter=N_ITER)
    ws = nmt.NmtWorkspace()
    ws.compute_coupling_matrix(f, f, bins)
    cl = np.asarray(ws.decouple_cell(nmt.compute_coupled_cell(f, f)))
    del f, ws
    return cl


rows = []
print(f"\\n{'nside':>6} {'Lmax':>6} {'march':>6} {'GM cold':>8} {'GM warm':>8} "
      f"{'field':>7} {'coup':>7} {'NM':>8} {'NM/GM':>6} {'rms':>9} {'devGB':>6}", flush=True)
for nside in NSIDES:
    lmax = 3 * nside - 1
    march = bool(
        PLATFORM == "gpu"
        and _march_v2.enabled(lmax + 1)
        and _use_pallas_sht(lmax + 1, 0)
        and fold_requested(nside, lmax + 1)
    )
    coup_route = "GEMM" if lmax >= _TT_QUADRATURE_LMAX else "3j"
    print(f"\\n=== nside={nside}  lmax={lmax}  march={march}  TT={coup_route} ===", flush=True)
    row = {
        "nside": nside, "lmax": lmax, "march": march, "tt": coup_route,
        "gm_cold": None, "gm_warm": None, "field": None, "coupling": None,
        "nm": None, "rms": None, "dev_gb": None, "error": None,
    }
    try:
        maps, mask = None, None
        lmax, maps, mask = make_maps(nside)
        bins_gm = gm.NmtBin.from_lmax_linear(lmax, NLB)
        t0 = time.perf_counter()
        cl_gm, stages0 = gmaster_once(mask, maps, bins_gm)
        row["gm_cold"] = time.perf_counter() - t0
        print(f"  GMaster cold {row['gm_cold']:.2f}s  stages {stages0}", flush=True)
        t0 = time.perf_counter()
        cl_gm, stages = gmaster_once(mask, maps, bins_gm)
        row["gm_warm"] = time.perf_counter() - t0
        row["field"] = stages["field"]
        row["coupling"] = stages["coupling"]
        stats = (dev.memory_stats() or {})
        peak = stats.get("peak_bytes_in_use")
        row["dev_gb"] = None if peak is None else peak / 1e9
        print(f"  GMaster warm {row['gm_warm']:.2f}s  field {stages['field']:.2f}s  "
              f"coupling {stages['coupling']:.2f}s  cell {stages['coupled_cell']:.2f}s  "
              f"decouple {stages['decouple']:.2f}s  device_peak={row['dev_gb']}", flush=True)
        # NaMaster at 4096 on 4 cores can take tens of minutes; still run it.
        bins_nm = nmt.NmtBin.from_lmax_linear(lmax, NLB)
        t0 = time.perf_counter()
        cl_nm = namaster_once(mask, maps, bins_nm)
        row["nm"] = time.perf_counter() - t0
        ratio = cl_gm[0] / np.where(cl_nm[0] == 0, np.nan, cl_nm[0]) - 1.0
        row["rms"] = float(np.nanstd(ratio))
        print(f"  NaMaster {row['nm']:.2f}s  NM/GM={row['nm']/row['gm_warm']:.2f}x  "
              f"rms(GM/NM-1)={row['rms']:.2e}", flush=True)
        del cl_gm, cl_nm, maps, mask, bins_gm, bins_nm
    except Exception as exc:  # noqa: BLE001 - keep the ladder going after OOM
        row["error"] = f"{type(exc).__name__}: {exc}"
        print("  FAILED:", row["error"], flush=True)
        traceback.print_exc()
    rows.append(row)
    gc.collect()
    try:
        jax.clear_caches()
    except Exception:
        pass

print("\\n=== ladder ===")
print(f"{'nside':>6} {'march':>6} {'GM warm s':>10} {'NM s':>10} {'NM/GM':>8} {'rms':>10} {'dev GB':>8}")
for row in rows:
    gm_s = "-" if row["gm_warm"] is None else f"{row['gm_warm']:.2f}"
    nm_s = "-" if row["nm"] is None else f"{row['nm']:.2f}"
    if row["gm_warm"] and row["nm"]:
        ratio_s = f"{row['nm']/row['gm_warm']:.2f}"
    else:
        ratio_s = "-"
    rms_s = "-" if row["rms"] is None else f"{row['rms']:.2e}"
    dev_s = "-" if row["dev_gb"] is None else f"{row['dev_gb']:.2f}"
    flag = "yes" if row["march"] else "no"
    extra = "" if not row["error"] else "  " + row["error"][:80]
    print(f"{row['nside']:6d} {flag:>6} {gm_s:>10} {nm_s:>10} {ratio_s:>8} {rms_s:>10} {dev_s:>8}{extra}")
